# E004: full-density recall probe (which retrieval channels recover the pairs the blocker misses?)

E001/E003 blocking recall was about 85-86% at full density. The 1% dev universe (about 99%) does not
predict it. This notebook measures recall **exactly at full density** but cheaply. It takes
`N_QUERIES` random train Source-1 records per country and searches each against the **complete**
S2/S3 pool of its country: same IDF, same df cap, same competition as the real run.

Channels (`src/ber/channels.py`):

| channel | what it tests |
|---|---|
| `base` | the E003 blocker (top-45 name+addr, plus top-10 name) |
| `base_wide` | same score, top-200: are misses *ranked too low* or *never scored*? |
| `addr_only` | address-only top-20: renamed or transliterated names |
| `name_noaddr` | name search among empty-address records only |
| `conj` | token-pair keys (addr x addr, name-skeleton x addr): frequent tokens become rare pairs |
| `bm25` | BM25 over the base keys with a much looser df cap (10k) |
| `rescue` | exact deterministic blocks (compact name, sorted skeleton, number set), small blocks only |
| `dense` | multilingual-e5-small (MIT) exact GPU search. Runs only if a GPU is attached |

Outputs: per-channel recall, **gain over base**, union recall, and the **RRF-truncated recall** at
M = 30/50/80/120/200 per query. Any recall lost by truncating is measured, not assumed. Also a
diagnosis of every base miss (rank, shared keys, native script, empty address), with which channel
recovered it.

**Settings:** Internet **On**. Accelerator: **GPU T4 x2** to include the dense channel, otherwise None.
Persistence: **Files**. Add the competition dataset as input. If you saved an earlier run's
`work/prepared/train/*.parquet` as a dataset, add it too: this skips about 30 min of normalization.

Runtime estimate (lexical channels, 10k queries per country): 30-60 min. Add about 45-60 min for dense.

In [ ]:
# 1. Config
N_QUERIES  = 10000    # sampled train Source-1 queries per country (recall SE ~0.3%)
CHANNELS   = "base,base_wide,addr_only,name_noaddr,conj,rescue,bm25"
RRF        = "base_wide,addr_only,name_noaddr,conj,rescue,bm25,dense"
DENSE      = "auto"   # "auto" = add the dense channel when a GPU is present; True / False to force
DENSE_MODEL = "intfloat/multilingual-e5-small"   # MIT, 118M params
REPO       = "https://github.com/Bexwane/AmazonMLchallenge.git"
CODE_DIR   = "/kaggle/working/ber"
WORK       = "/kaggle/working/work"
PROBE_OUT  = "/kaggle/working/probe_E004"

In [ ]:
# 2. Locate the dataset (and an optional saved prepared cache) under /kaggle/input
import glob, os
hits = glob.glob("/kaggle/input/**/train/train_source1.tsv", recursive=True)
assert hits, "Dataset not found: add the dataset with train/ and test/ folders as notebook input"
DATA = os.path.dirname(os.path.dirname(hits[0]))
print("DATA =", DATA)
PREP = f"{WORK}/prepared/train"
if not os.path.exists(f"{PREP}/s23.parquet"):
    saved = glob.glob("/kaggle/input/**/prepared/train/s23.parquet", recursive=True)
    if saved:
        os.makedirs(PREP, exist_ok=True)
        for f in ("s1.parquet", "s23.parquet"):
            os.symlink(os.path.join(os.path.dirname(saved[0]), f), f"{PREP}/{f}")
        print("reusing saved prepared cache:", os.path.dirname(saved[0]))
print("prepared cache present:", os.path.exists(f"{PREP}/s23.parquet"))
!free -g; nproc; nvidia-smi -L 2>/dev/null || echo "no GPU"

In [ ]:
# 3. Code + dependencies + unit tests
!rm -rf {CODE_DIR} && git clone -q {REPO} {CODE_DIR} && cd {CODE_DIR} && git log --oneline -1
!pip install -q rapidfuzz==3.14.6
import sys; sys.path.insert(0, f"{CODE_DIR}/src")
import torch
USE_DENSE = torch.cuda.is_available() if DENSE == "auto" else bool(DENSE)
if USE_DENSE:
    !pip install -q sentence-transformers
    CHANNELS += ",dense"
print("channels:", CHANNELS)
!cd {CODE_DIR} && python -m pytest -q tests

In [ ]:
# helper: run a script with live log streaming
import subprocess, time
def run(*args):
    t = time.time()
    env = {**os.environ, "PYTHONPATH": "src", "BER_DENSE_MODEL": DENSE_MODEL}
    p = subprocess.Popen(["python", *args], cwd=CODE_DIR, env=env,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    assert p.wait() == 0, f"failed (exit {p.returncode}); -9 means out of memory"
    print(f"--- done in {(time.time() - t) / 60:.1f} min")

In [ ]:
# 4. Normalize train once (cached; skipped if the prepared parquet already exists)
run("-c", f"from ber.prepare import load_prepared; s1, s23 = load_prepared({DATA!r}, 'train', {WORK!r}); print(len(s1), len(s23))")
!du -shL {WORK}/prepared/train

In [ ]:
# 5. Probe: one process per country, so memory is released between countries
import pandas as pd
countries = sorted(pd.read_parquet(f"{WORK}/prepared/train/s1.parquet", columns=["country"])["country"].unique())
print(countries)
for c in countries:
    run("scripts/recall_probe.py", "--data", DATA, "--work", WORK, "--n", str(N_QUERIES),
        "--channels", CHANNELS, "--rrf", RRF, "--countries", c, "--out", PROBE_OUT)
    !free -g | head -2

In [ ]:
# 6. Summary: per-channel recall, gain over base, union, RRF truncation, and the >= 98% gate
import json
rep = json.load(open(f"{PROBE_OUT}/probe.json"))
rows = []
for c, r in rep.items():
    for ch in CHANNELS.split(","):
        if ch in r:
            rows.append({"country": c, "channel": ch, **{k: r[ch].get(k) for k in
                         ("recall", "recall@10", "recall@45", "gain_over_base", "pairs_per_q", "sec")}})
    rows.append({"country": c, "channel": "UNION", "recall": r["union"]["recall"], "pairs_per_q": r["union"]["pairs_per_q"]})
pd.set_option("display.width", 200)
display(pd.DataFrame(rows).round(4))
display(pd.DataFrame({c: r["rrf"] for c, r in rep.items()}).drop("channels"))
display(pd.DataFrame({c: r["base_miss_taxonomy"] for c, r in rep.items()}).round(3))
tp = {c: r["true_pairs"] for c, r in rep.items()}
tot = sum(tp.values())
def pooled(get):
    return sum(get(r) * tp[c] for c, r in rep.items()) / tot
res = {"base": pooled(lambda r: r["base"]["recall"]), "UNION": pooled(lambda r: r["union"]["recall"])}
for m in (50, 80, 120, 200):
    res[f"rrf@{m}"] = pooled(lambda r: r["rrf"][f"recall@{m}"])
for k, v in res.items():
    print(f"{k:10s} pooled recall = {v:.4f}   {'PASS' if v >= 0.98 else 'below'} 98% gate")

In [ ]:
# 7. Base misses: which channel recovers them, and what the unrecovered ones look like
m = pd.read_csv(f"{PROBE_OUT}/misses.tsv", sep=chr(9), keep_default_na=False)
hit_cols = [c for c in m.columns if c.startswith("hit_") and c != "hit_base"]
for c in hit_cols:
    m[c] = m[c].astype(str) == "True"
m["any"] = m[hit_cols].any(axis=1)
print("base misses:", len(m))
display(m.groupby("country")[hit_cols + ["any"]].mean().round(3))
cols = ["country", "s1_name", "s1_addr", "r_name", "r_addr", "comb_rank", "name_cos", "addr_cos"]
display(m.loc[~m["any"], cols].sample(min(40, int((~m["any"]).sum())), random_state=0))

In [ ]:
# 8. Pack the small result files: download probe_E004.tgz (probe.json + misses.tsv) and send it back.
# Tip: "Save Version" with Persistence=Files keeps work/prepared, so the next notebook can reuse it as input.
!cd /kaggle/working && tar czf probe_E004.tgz -C {PROBE_OUT} . && ls -la probe_E004.tgz {PROBE_OUT}